# 01 — Putting the data together
What this notebook does: It gets the data ready. It doesn't analyse anything.

The goal of Mini Project 2: Work out which neighbourhoods in New Zealand have crowded housing, and what else about those neighbourhoods might help explain it.

What happens here, step by step:

Load the data. We read the 2023 Census household figures from Stats NZ. They're grouped by small neighbourhood areas called SA2s, which are roughly suburb-sized.
Give the columns proper names. The raw file labels its 261 columns with codes like VAR_4_n, which tell you nothing. Stats NZ provides a guide that says what each code means. We use that guide only to rename the columns. We don't merge it into the data.
Work out the thing we want to predict: crowding. For each area we calculate how crowded its households are. This is called the "target".
Build one tidy table. It has one row per neighbourhood (SA2).
Why we add more data later: If we only used the household table to explain crowding, we'd be going in circles. The explanations would come from the same numbers that define crowding. So the next step is to add other Census tables, such as income or ethnicity, to each neighbourhood. These give us independent clues about what goes with crowding.

What this notebook produces: Two files that the later notebooks use.

*outputs/analysis_frame.csv* is the finished table, one row per neighbourhood.
*outputs/frame_meta.json* is a small note recording how "crowded" was defined and the cut-off used. That way the definition is written down with the data and nobody has to remember it.
How to run it: Start from a fresh copy of the project and run the notebook from top to bottom. It should work with no extra setup.

## Setup

Setting up: file locations, a forgiving file reader, and a number reader
What this section does: It sets up three basic tools the notebook needs before it can load anything.

File locations. It tells the notebook where the input data lives and where to save its results.
A forgiving file reader. Files from different sources store text in slightly different ways, and a strict reader can fail on special characters, such as the macrons in Māori place names. This reader tries more than one way of reading the file, so it doesn't stop at the first problem.
A number reader. Stats NZ's tables don't always hold plain numbers. Some cells hold a placeholder code that means "hidden" or "not available". The number reader turns real values into numbers and treats the placeholders as missing.
What changed: The number reader and its list of placeholder codes used to be written inside this notebook. They now come from a shared file called mp2, which all the project's notebooks use.

Why: The placeholder codes are a feature of the Stats NZ data, not of this notebook, so they belong in one shared place. This also fixes a real gap. When we worked on notebook 02, we found a second placeholder code, -997. The old copy in this notebook didn't know about it, so those values could have been read as real numbers. With the shared version, every notebook now catches the same codes. Any future fix only has to be made once.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

sys.path.append("../src")
import mp2
from mp2 import read_measure, SENTINEL_FLOOR

# The download folder. One place, so the absolute path appears once.
DATA = Path(r"D:\dsai\dsai_projects\Mini-Project-2-Data")
OUT  = mp2.OUT
OUT.mkdir(exist_ok=True, parents=True)

HOUSEHOLDS_FILE = DATA / "2023-census-totals-by-topic-for-households-by-statistical-ar.csv"
LOOKUP_FILE     = DATA / "2023_census_totals_by_topic_for_households_by_sa2_lookup_table.csv"
SA2_SHAPE       = DATA / "statistical-area-2-2023-generalised.shp"

YEAR = mp2.CENSUS_YEAR


def load_raw(path):
    """Read a Stats NZ export. Tries UTF-8, then the common Windows encoding."""
    for enc in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(path.name)


def slug(value):
    """Lower-case, underscore-separated column name from a Stats NZ label."""
    import re
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


print("data folder   :", DATA)
print("households ok :", HOUSEHOLDS_FILE.exists())
print("lookup ok     :", LOOKUP_FILE.exists())
print("sentinel floor:", SENTINEL_FLOOR)

data folder   : D:\dsai\dsai_projects\Mini-Project-2-Data
households ok : True
lookup ok     : True
sentinel floor: -990


## Loading and renaming

Loading the lookup guide and renaming the columns
What this section does: It reads Stats NZ's guide to the column codes and uses it to give every column a readable name as the data comes in.

Pick out the right rows of the guide. Stats NZ keeps the 2013, 2018 and 2023 Census descriptions in one combined file. We keep only the 2023 rows, because the older years use the same codes for different things. We also keep only the rows that describe the data columns (the ones starting with VAR_) and ignore the rest.
Rename on the way in. The columns get their readable names as the data is loaded, not afterwards. This means the codes never appear anywhere else in the project. Every later step uses the plain names, so the work is easier to follow and there is less risk of picking the wrong column by mistake.
Why this matters: If we had used the whole combined guide, a code could have been matched to its 2013 or 2018 meaning instead of its 2023 one. That would give a column the wrong name without any error. Filtering first prevents that.

In [2]:
households = load_raw(HOUSEHOLDS_FILE)
lookup     = load_raw(LOOKUP_FILE)

lookup_2023 = lookup[(lookup["Year"] == YEAR) &
                     lookup["Column_name"].astype(str).str.startswith("VAR_")].copy()

lookup_2023["readable_name"] = [
    f"{slug(topic)}_{slug(category)}"
    for topic, category in zip(lookup_2023["Variable1"], lookup_2023["Variable1_category"])
]
rename_columns = dict(zip(lookup_2023["Column_name"], lookup_2023["readable_name"]))

keep = [c for c in rename_columns if c in households.columns]
analysis = (households[["SA22023_V1_00", "SA22023_V1_00_NAME"] + keep]
            .rename(columns={"SA22023_V1_00": "sa2_code",
                             "SA22023_V1_00_NAME": "sa2_name"})
            .rename(columns=rename_columns))

# Coerce every measure column once, here. read_measure returns float64
# unconditionally, so no later .astype("float64") patch is needed.
value_columns = [c for c in analysis.columns if c not in ("sa2_code", "sa2_name")]
analysis[value_columns] = analysis[value_columns].apply(read_measure)

print("rows, columns :", analysis.shape)
print("rate dtype check — all numeric:",
      bool(analysis[value_columns].dtypes.eq("float64").all()))
print("usable SA2 codes:", analysis["sa2_code"].notna().sum(), "of", len(analysis))

rows, columns : (2395, 89)
rate dtype check — all numeric: True
usable SA2 codes: 2395 of 2395


## Checking that the crowding numbers add up
What this section does: It checks the crowding figures for internal consistency before anything is built on them. Making sure I trust the results and numbers I am seeing.

The two checks:

Crowded households. The total number of crowded households should equal the "severely crowded" count plus the "moderately crowded" count.
All households. The total number of households should equal the households that answered the question plus the households that didn't answer.
Small differences are fine. Stats NZ rounds every count on its own, so the parts can be slightly off from the total. The check allows for this.

A negative total is a red flag. Some cells in the Stats NZ tables hold special codes instead of real counts. If one of these gets read as a number, it shows up as a negative total. The check stops the notebook and says so.

The codes we know about:

-999: The area's population was too small to show the answer. Stats NZ hides these cells on purpose to protect people's privacy. This is called suppression.
-997: There was no response. We found this one later.
Why it stays: This check is how we first discovered the -999 problem. It is also what led us to the -997 code. Keeping it means any new hidden code in future data will be caught straight away instead of quietly skewing the results.


In [3]:
CROWD = {
    "crowded":      "household_crowding_index_crowded",
    "severe":       "household_crowding_index_two_or_more_bedrooms_needed_severely_crowded",
    "moderate":     "household_crowding_index_one_bedroom_needed_crowded",
    "not_crowded":  "household_crowding_index_not_crowded",
    "not_stated":   "household_crowding_index_not_stated",
    "total":        "household_crowding_index_total",
    "total_stated": "household_crowding_index_total_stated",
}

totals = {name: analysis[col].sum() for name, col in CROWD.items()}
for name, value in totals.items():
    print(f"  {name:14s} {value:>12,.0f}")

print()
print(f"crowded  vs severe+moderate : {totals['crowded'] - totals['severe'] - totals['moderate']:+,.0f}")
print(f"total    vs stated+notstated: {totals['total'] - totals['total_stated'] - totals['not_stated']:+,.0f}")

assert totals["crowded"] > 0, "crowded is not positive - is a sentinel getting through?"
assert totals["total"] > 0, "total is negative - a numeric sentinel survived read_measure"
print()
print("No total is negative, and both identities hold to within rounding.")

  crowded             105,633
  severe               30,861
  moderate             74,730
  not_crowded       1,598,841
  not_stated           75,984
  total             1,780,530
  total_stated      1,704,474

crowded  vs severe+moderate : +42
total    vs stated+notstated: +72

No total is negative, and both identities hold to within rounding.


No count is a minus number. A household count can't be below zero. In the Stats NZ data, some cells hold a special code (like -999) instead of a real count. If one of those had been treated as a real number, there would be a negative. There were none.
The numbers add up. Two sums were checked:
Crowded households = severely crowded + moderately crowded
All households = households that answered + households that didn't

## The target: what we're trying to predict
What this section does: It decides what "crowded" means and turns it into a single yes/no label for each neighbourhood.

What "crowded" means. A household is crowded if it needs one or more extra bedrooms to house everyone properly.

Step 1: Work out a crowding rate for each neighbourhood. We take the number of crowded households and divide it by the households that answered the question. Some households skip the question, and we leave them out of the division.

Why? If we divided by all households, an area where many people skipped the question would look less crowded than it really is. Those skipped households would be counted as "not crowded" when we simply don't know.

Example: An area has 100 households. 80 answered and 20 skipped. 16 of those who answered are crowded. The fair rate is 16 out of 80, or 20%. Dividing by all 100 would give 16%, which is too low.

Step 2: Measure whole neighbourhoods, not individual homes. The rate describes an entire neighbourhood (an SA2). It says "20% of households here are crowded". It does not say whether any one house is crowded.

Step 3: Flag the top 25%. We label the most crowded quarter of neighbourhoods as "high crowding" and everything else as "not high".

Why a quarter? If we set a strict high/low cut-off, very few neighbourhoods would qualify. With so few "high" examples, the scores we use to judge how well the analysis works (such as PR-AUC) become meaningless. Taking the top quarter gives us a reasonable-sized group to work with.

It also matches how the client uses the results. They don't need a strict pass/fail label. They want to rank neighbourhoods so they know which to look at first, and "the top quarter" fits that.

To summarise: each neighbourhood gets a fair crowding rate, and the most crowded 25% are flagged as the priority group.

## Keeping the crowding rate as a proper number
What this section does: It explains a small coding choice. When the notebook divides crowded households by responding households, the result has to be stored as a true number. It was once being stored as something else.

The problem. When a count was missing, the code used a marker called pd.NA. This is a "nothing here" placeholder, but it behaves differently from ordinary numbers. When the division ran, it pulled the whole column into a mixed, catch-all format. The column then held items that looked like numbers but weren't treated as numbers.

Why it was hard to spot. The crowding rates looked correct when printed to four decimal places. The 25% cut-off (the line between "high" and "not high") was calculated from this column, so it looked right too. Underneath, though, the column was the wrong type, and that could have produced wrong results later without any warning.

The fix. The code now uses np.nan, a different "nothing here" marker that is built to work with numbers. With it, the division always produces a proper column of decimal numbers (float64). Missing values stay as blanks, and everything else is an ordinary number.

The rule to keep: use np.nan, never pd.NA, for missing values in this calculation.

In one sentence: the missing-value marker was changed so the crowding rates are stored as real numbers, which makes the top-25% cut-off trustworthy.

In [4]:
analysis["crowded_n"]     = analysis[CROWD["crowded"]]
analysis["crowding_base"] = analysis[CROWD["total_stated"]]
analysis["crowding_rate"] = (
    analysis["crowded_n"]
    / analysis["crowding_base"].replace(0, np.nan)
).astype("float64")

assert analysis["crowding_rate"].dtype == "float64", analysis["crowding_rate"].dtype
assert not analysis["crowding_rate"].isna().all(), "every rate is missing"
assert analysis["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

# Top-quartile threshold, computed once and written to frame_meta.json so
# notebooks 02-05 reuse it instead of each recalculating.
threshold = float(analysis["crowding_rate"].quantile(0.75))
analysis["crowded_top_quartile"] = (
    analysis["crowding_rate"] >= threshold
).astype("float64")

# Areas with no usable rate are unknown, not negative. Leaving them NaN keeps
# them out of training rather than silently teaching the model they are fine.
analysis.loc[analysis["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

n_known   = int(analysis["crowding_rate"].notna().sum())
n_positive = int(analysis["crowded_top_quartile"].sum())
n_unknown = int(analysis["crowding_rate"].isna().sum())

print(f"rate dtype             : {analysis['crowding_rate'].dtype}")
print(f"top-quartile threshold : {threshold:.4f}  ({threshold:.2%})")
print()
print(analysis["crowding_rate"].describe().round(5).to_string())
print()
print(f"SA2s with a usable rate: {n_known:,} of {len(analysis):,}")
print(f"without a usable rate  : {n_unknown:,}")
print(f"positive class         : {n_positive:,} SA2s")
print(f"base rate              : {n_positive / n_known:.2%}  (of areas with a rate)")
print()
print(f"predicting the majority class every time scores "
      f"{max(n_positive / n_known, 1 - n_positive / n_known):.2%} accuracy "
      f"and 0.00 recall on the crowded class.")

rate dtype             : float64
top-quartile threshold : 0.0775  (7.75%)

count    2283.00000
mean        0.06409
std         0.06505
min         0.00000
25%         0.02729
50%         0.04494
75%         0.07746
max         1.00000

SA2s with a usable rate: 2,283 of 2,395
without a usable rate  : 112
positive class         : 572 SA2s
base rate              : 25.05%  (of areas with a rate)

predicting the majority class every time scores 74.95% accuracy and 0.00 recall on the crowded class.


## Sanity checks

Testing the validity of the data and features.


In [5]:
assert analysis["sa2_code"].is_unique, "SA2 codes must be unique"
assert analysis["sa2_code"].notna().all(), "no missing SA2 codes"
assert analysis["crowded_n"].min() >= 0, "negative crowding counts"
assert analysis["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

print(f"SA2s                 : {len(analysis):,}")
print(f"unique SA2 codes     : {analysis['sa2_code'].nunique():,}")
print(f"code length          : {sorted(analysis['sa2_code'].astype(str).str.len().unique())}")
print(f"crowded count range  : {analysis['crowded_n'].min():.0f} to {analysis['crowded_n'].max():.0f}")
print(f"crowding rate range  : {analysis['crowding_rate'].min():.4f} to {analysis['crowding_rate'].max():.4f}")
print()
print("All checks passed.")

SA2s                 : 2,395
unique SA2 codes     : 2,395
code length          : [np.int64(6)]
crowded count range  : 0 to 288
crowding rate range  : 0.0000 to 1.0000

All checks passed.


## Where the hidden values show up
What this section does: It counts how many neighbourhoods have hidden numbers in their data, and checks whether those are mostly the small ones.

The claim being tested. As state in the project's README, to maintain anonimity for census respondents some SA2 or neighbourhood values are hiddne. These hidden values mostly occur in small neighbourhoods. Stats NZ hides a number when the group is so small that someone might be identified. This section counts the cases, so the claim made in the README rests on evidence rather than an assumption I made based on my knowledge/assumption of the census data.

What "hidden" means here. Stats NZ uses special placeholder codes instead of real counts:

-999: the area was too small to show the answer. Stats NZ hides it on purpose to protect privacy. This is called suppression.
-997: there was no response.
What was fixed. The count now uses a shared checking tool, mp2.flag_suppressed, which recognises both codes. The old version only looked for -999. It missed every -997, so it undercounted the hidden values.

Why it matters. The old count could have made the problem look smaller than it is. With both codes included, the numbers behind the README's claim are complete.

This section counts every neighbourhood with a hidden or missing value, using a tool that knows both codes, to show whether they cluster in small areas.

In [6]:
raw = load_raw(HOUSEHOLDS_FILE)
raw_var_columns = [c for c in raw.columns if str(c).startswith("VAR_4_")]

# Count every suppressed or non-applicable cell, not just the -999 sentinel.
analysis["suppressed_n"] = raw[raw_var_columns].apply(mp2.flag_suppressed).sum(axis=1)

lost = analysis["crowding_rate"].isna()

print(f"SA2s without a crowding rate: {lost.sum()} of {len(analysis)}")
print()
print("suppressed cells per SA2:")
print(analysis["suppressed_n"].describe().round(2).to_string())
print()
print(f"mean suppressed cells - SA2s kept : {analysis.loc[~lost, 'suppressed_n'].mean():.1f}")
print(f"mean suppressed cells - SA2s lost : {analysis.loc[lost,  'suppressed_n'].mean():.1f}")
print()
print("suppressed cells in the lost areas  :", int(analysis.loc[lost,  "suppressed_n"].sum()))
print("suppressed cells in all other areas :", int(analysis.loc[~lost, "suppressed_n"].sum()))
print()
print("correlation, suppression vs crowding rate:",
      round(analysis["suppressed_n"].corr(analysis["crowding_rate"]), 3))

SA2s without a crowding rate: 112 of 2395

suppressed cells per SA2:
count    2395.00
mean       20.95
std        48.81
min         8.00
25%         8.00
50%         8.00
75%         8.00
max       231.00

mean suppressed cells - SA2s kept : 10.7
mean suppressed cells - SA2s lost : 229.1

suppressed cells in the lost areas  : 25654
suppressed cells in all other areas : 24516

correlation, suppression vs crowding rate: 0.116


Every area has at least 8 hidden cells. The minimum, the 25th percentile, the median and the 75th percentile are all 8. So 8 hidden cells is normal and seems to be built into the table. Most areas sit at 8, and a few have far more (up to 231).
Lost areas have far more hidden cells. Areas that kept their crowding rate average 10.7 hidden cells. Lost areas average 229.1, which is about 21 times more.
The lost areas hold about half of all hidden cells. They have 25,654 hidden cells, against 24,516 in the other 2,283 areas. So 112 areas, or under 5% of the total, hold about 51% of the hidden values.
Correlation of 0.116. This compares the number of hidden cells with the crowding rate. A score of 0 means no link, and 1 means a perfect link. A score of 0.116 is a very weak link. Hidden values don't strongly push crowding rates up or down in the areas that kept a rate.

## The ranking is noisy at the top

The two areas with the highest crowding rate have three households between
them. This matters for a client who prioritises by ordering, so it is measured
here rather than left as a caveat.


In [7]:
print("top 10 most crowded SA2s (unsorted by size):")
print(analysis.nlargest(10, "crowding_rate")[
    ["sa2_name", "crowded_n", "crowding_base", "crowding_rate"]
].to_string(index=False))
print()
for cutoff in (20, 50, 100):
    small = analysis["crowding_base"] < cutoff
    print(f"SA2s under {cutoff:>3} households: {small.sum():>3}  "
          f"rate range {analysis.loc[small, 'crowding_rate'].min():.3f} to "
          f"{analysis.loc[small, 'crowding_rate'].max():.3f}")
print()

top 10 most crowded SA2s (unsorted by size):
                          sa2_name  crowded_n  crowding_base  crowding_rate
                         Fiordland        3.0            3.0       1.000000
Islands Thames-Coromandel District        3.0            3.0       1.000000
               Takanini Industrial        3.0            6.0       0.500000
                      Wesley South        6.0           12.0       0.500000
                     Rongomai West      186.0          447.0       0.416107
                          Ferguson      234.0          570.0       0.410526
                        Ōtara West      243.0          612.0       0.397059
                      Māngere West      264.0          675.0       0.391111
                     Harania North      279.0          720.0       0.387500
                        Ōtara East      288.0          753.0       0.382470

SA2s under  20 households:  21  rate range 0.000 to 1.000
SA2s under  50 households:  51  rate range 0.000 to 1.000
SA

## What the results show

### Top 10 most crowded SA2s, ranked by rate alone

| Rank | Area | Crowded | Answered | Rate |
|:--|:--|--:|--:|--:|
| 1 | Fiordland | 3 | 3 | 100% |
| 2 | Islands Thames-Coromandel District | 3 | 3 | 100% |
| 3 | Takanini Industrial | 3 | 6 | 50% |
| 4 | Wesley South | 6 | 12 | 50% |
| 5 | Rongomai West | 186 | 447 | 42% |
| 6 | Ferguson | 234 | 570 | 41% |
| 7 | Ōtara West | 243 | 612 | 40% |
| 8 | Māngere West | 264 | 675 | 39% |
| 9 | Harania North | 279 | 720 | 39% |
| 10 | Ōtara East | 288 | 753 | 38% |

- **The top four are tiny.** They have 3 to 12 answering households each. Their high rates come from a handful of households, not a pattern.
- **Places 5 to 10 are the real signal.** Each has hundreds of crowded households, and the rates are still high at around 40%. These are the areas a client would want to see first.
- **The tiny areas push the real ones down.** Without them, the six areas from Rongomai West to Ōtara East would be ranked 1st to 6th.

### Small areas swing to extremes

| Size of area | Number of areas | Rate range |
|:--|--:|--:|
| Under 20 households | 21 | 0% to 100% |
| Under 50 households | 51 | 0% to 100% |
| Under 100 households | 82 | 0% to 100% |

Even areas with up to 100 households cover the whole range, from nobody crowded to everybody crowded. A rate from one of these areas says almost nothing. Together, the 82 areas under 100 households are only about 3.4% of the 2,395 areas.

## What this means for the client

- **The problem is concentrated.** It is a few dozen small areas, not the whole ranking. After the first four places, the list looks sensible.
- **A minimum household count would fix it.** Ranking only areas with 100 or more answering households would remove all four noisy entries. Every area from place 5 down has well over 400. A cut-off of 50 also works for the top 10, so the choice matters mostly further down the list.
- **The top-25% flag is affected too.** Small areas with a high rate are counted as "high crowding" in the target. This adds some noise to the 572 flagged areas. It is a smaller problem than at the top of the ranking, because the flag is a broad band, not an exact order.
- **Fiordland and Islands Thames-Coromandel District are real, just tiny.** They are mostly national park and islands, so three households is plausible. They are not data errors.

**In one sentence:** the top four spots are filled by areas with 3 to 12 households, so the ranking is only trustworthy from place 5 down unless a minimum household count is applied.


## Caution: The top of the ranking is dominated by tiny denominators. ##
A 3-household area scoring 1.000 outranks Ōtara West (612 households) purely due to sample size.
We need to apply a size floor to the shortlist.

## Writing the analysis table
What this section does: It saves the finished table as a file, along with a small note explaining how it was built. This is the hand-off. Notebooks 02 to 05 start from these two files and don't repeat any of the work above.

The two files:

 - outputs/analysis_frame.csv is the table, with one row per neighbourhood (SA2). It's a plain spreadsheet-style file that any program can open.
 - outputs/frame_meta.json is a short note that travels with the table. It records the decisions made here: how "crowded" was defined and the 7.75% cut-off for the top 25%.
Why the note matters. The cut-off and the definition of crowding are written down next to the data. Later notebooks read them from the note, not from memory, and they don't each work out their own version. If someone asks in six months what "high crowding" meant, the answer is in the file.

Why no map data is in the table. Map boundaries (the shapes of each neighbourhood) are left out on purpose.

 - Notebook 02 needs maps, so it reads the boundary file itself when it needs it.
 - Notebooks 03 to 05 don't draw maps. A plain table means they don't need the specialist map software installed, which keeps them simpler and quicker to run.
Why the SA2 code is kept. The code is the neighbourhood's unique ID, like a street address. It's how later notebooks attach new Census tables (income, ethnicity and so on) to the right row. Without it, they couldn't match the new data to the neighbourhoods.

This step saves the clean table and a note of the rules used, so every later notebook starts from the same data and the same definition of "crowded".

In [8]:
# Fail here rather than three notebooks later if a sentinel survived.
mp2.assert_no_sentinels(analysis)

analysis.to_csv(OUT / "analysis_frame.csv", index=False)
print()
print("written:", OUT / "analysis_frame.csv")

frame_meta = {
    "year": YEAR,
    "n_sa2": int(len(analysis)),
    "source": "Stats NZ 2023 Census totals by topic for households, by SA2",
    "licence": "CC BY 4.0, Statistics New Zealand",
    "crowding_definition": "one or more bedrooms needed (crowded)",
    "crowding_denominator": "total stated, excludes not stated",
    "target": "crowded_top_quartile",
    "top_quartile_threshold": threshold,
    "sa2_with_rate": n_known,
    "sa2_without_rate": n_unknown,
    "positive_class": n_positive,
    "base_rate": round(n_positive / n_known, 6),
    "suppressed_cells": int(analysis["suppressed_n"].sum()),
    "sentinel_floor": SENTINEL_FLOOR,
}

(OUT / "frame_meta.json").write_text(json.dumps(frame_meta, indent=2))
print()
print(json.dumps(frame_meta, indent=2))

No sentinels at or below -990 in 93 numeric columns.

written: C:\Users\akdux\dsai-mini-project-2\outputs\analysis_frame.csv

{
  "year": 2023,
  "n_sa2": 2395,
  "source": "Stats NZ 2023 Census totals by topic for households, by SA2",
  "licence": "CC BY 4.0, Statistics New Zealand",
  "crowding_definition": "one or more bedrooms needed (crowded)",
  "crowding_denominator": "total stated, excludes not stated",
  "target": "crowded_top_quartile",
  "top_quartile_threshold": 0.07746478873239436,
  "sa2_with_rate": 2283,
  "sa2_without_rate": 112,
  "positive_class": 572,
  "base_rate": 0.250548,
  "suppressed_cells": 50170,
  "sentinel_floor": -990
}


## Notes for the README

Stats NZ applies fixed random rounding to base 3 and suppresses sensitive cells
below six. Two numeric sentinels appear in the totals-by-topic tables: `-999`
for a suppressed cell and `-997` for a category not collected or not
applicable. Both are mapped to missing by `mp2.read_measure`. Notebook 01's
reconciliation check caught `-999`; notebook 02 caught `-997`, in a column whose
every value was the sentinel — which is why a missing-value count reported zero.

Suppression is heavily concentrated: the median SA2 carries none, while 112
areas (4.7% of the total) average 221 suppressed cells each against 2.7 for the
areas retained. Those 112 areas lose the crowding rate entirely and are excluded
from the model, so the results describe the areas where the census can be
published — systematically the larger ones.
